# XM655 channel estimation

Measure every channel of a two system link with a CW tone.

A system is a transmit set and a receive set: system 0 is `DACS_0` / `ADCS_0`,
system 1 is `DACS_1` / `ADCS_1`. Four matrices come out, named
`H<receiver><transmitter>`:

| | from | to | what it is |
|---|---|---|---|
| `H00` | `DACS_0` | `ADCS_0` | system 0 hearing itself, its self interference |
| `H11` | `DACS_1` | `ADCS_1` | system 1 hearing itself |
| `H01` | `DACS_1` | `ADCS_0` | 1 reaching 0 |
| `H10` | `DACS_0` | `ADCS_1` | 0 reaching 1 |

One sweep gives all four: every DAC of both systems is excited and all 16 ADCs
are kept from every burst.

## 1. Parameters

Everything that gets tuned. Nothing outside this cell should need editing.

The NCO sets the RF frequency and is per *tile*, so all four DACs of a tile share
one waveform and differ only in phase and gain. The ADC samples at 2.5 GSPS, so a
4900 MHz signal folds down to `5000 - 4900 = 100 MHz`. Rule of thumb:
**`ADC_NCO = 5000 - DAC_NCO`**, and the tone comes back at its own frequency.
Do not go much above 4900 MHz - the receive mixer also makes an image twice the
fold away, and if the fold is small that image lands in band and ruins the
signal.

The four lists hold `overlay.dac[]` and capture indices, not RF channel labels.
On this board tile 2 reaches the antennas, so system 0 transmits on DACs 8..11;
system 1 is put on tile 3 - set it to whatever the second node is actually wired
to.

In [ ]:
from lib.config_parser import load_config

CFG = load_config()

# --- system 0: transmit set and receive set ---
DACS_0 = CFG["systems"]["dacs_0"]    # overlay.dac[] indices, tile 2
ADCS_0 = CFG["systems"]["adcs_0"]    # capture indices of its receivers, tile 1

# --- system 1: the node on the other side of the link ---
DACS_1 = CFG["systems"]["dacs_1"]    # overlay.dac[] indices, tile 3
ADCS_1 = CFG["systems"]["adcs_1"]    # capture indices of its receivers, tile 0

# --- rf ---
DAC_NCO  = CFG["rf"]["dac_nco"]      # MHz, per tile -> TX lands at 4900
DAC_ZONE = CFG["rf"]["dac_zone"]     # Nyquist zone, per tile
ADC_NCO  = CFG["rf"]["adc_nco"]      # MHz, per tile = 5000 - DAC_NCO
ADC_ZONE = CFG["rf"]["adc_zone"]     # the fold is in an even zone

# --- rates: fixed by the bitstream, do not change ---
DAC_SR = CFG["board"]["dac_sr"]      # DAC baseband rate = 10 GSPS / 10 (C2R eats one x2)
ADC_SR = CFG["board"]["adc_sr"]      # ADC baseband rate = 2.5 GSPS / 10 decimation
N_CH   = CFG["board"]["n_ch"]        # RF channels on the XM655
N_TILE = CFG["board"]["n_tile"]      # ADC and DAC tiles

# --- cw tone played during the estimation ---
CW_TONE_MHZ = CFG["signal"]["tone_mhz"]  # baseband tone, offset down from the tile NCO
CW_AMP      = CFG["signal"]["amp"]       # 14 bit DAC: +16383 / -16384

# --- estimation method ---
HADAMARD_EST     = False   # False: one DAC at a time. True: all DACs, Hadamard signs
INVERT_PHASE_DEG = 179.99  # the -1 of a Hadamard row - the SoC rejects exactly 180

# --- one capture ---
N_CAP       = CFG["capture"]["n_cap"]        # samples per channel
TRIG_HOLD_S = CFG["capture"]["trig_hold_s"]  # trig_cap must stay high for a whole capture window


## 2. Verify the parameters

Catch a bad settings cell here, before anything touches the hardware. The checks
that matter most are the index ones: a DAC or an ADC listed under two systems
puts the same measurement into two matrices, and nothing downstream can tell.

The rest is the usual range checking - indices inside the hardware, a tone that
fits the band, a capture size the converters can deliver.

`snap_tone_to_fft_bin()` is the odd one out: it does not reject anything, it
**overrides** `CW_TONE_MHZ`, moving it to the nearest exact FFT bin. The whole
estimate is one FFT value read at the tone bin, so a tone sitting between bins
spreads over its neighbours and the phase read off the peak is not the phase of
the channel. From here on `CW_TONE_MHZ` is the snapped value.

The bin grid is `ADC_SR / N_CAP`. With the usual 65536 complex sample player the
DAC loop repeats on that same grid, so one snap also keeps the transmitted
waveform continuous across the wrap - worth re-checking if the player length
ever changes.

In [ ]:
import numpy as np

from lib.common_functions import snap_tone_to_fft_bin

def validate_system_lists():
    """Every system needs something to transmit with and something to hear on."""
    lists = {"DACS_0": DACS_0, "ADCS_0": ADCS_0, "DACS_1": DACS_1, "ADCS_1": ADCS_1}
    for name, indices in lists.items():
        if len(indices) == 0:
            raise ValueError("%s is empty - a system needs at least one element" % name)

def validate_channel_indices():
    """Every index exists on the XM655, and belongs to exactly one system."""
    for name, indices in (("DACS_0", DACS_0), ("ADCS_0", ADCS_0),
                          ("DACS_1", DACS_1), ("ADCS_1", ADCS_1)):
        if not set(indices) <= set(range(N_CH)):
            raise ValueError("%s holds indices outside 0..%d" % (name, N_CH - 1))
        if len(set(indices)) != len(indices):
            raise ValueError("%s lists the same index twice" % name)
    shared_dacs = set(DACS_0) & set(DACS_1)
    if shared_dacs:
        raise ValueError("DACs %s are listed under both systems" % sorted(shared_dacs))
    shared_adcs = set(ADCS_0) & set(ADCS_1)
    if shared_adcs:
        raise ValueError("ADCs %s are listed under both systems" % sorted(shared_adcs))

def validate_converter_settings():
    """NCO and Nyquist zone tables are per tile, and the folds must add up."""
    for name, table in (("DAC_NCO", DAC_NCO), ("DAC_ZONE", DAC_ZONE),
                        ("ADC_NCO", ADC_NCO), ("ADC_ZONE", ADC_ZONE)):
        if len(table) != N_TILE:
            raise ValueError("%s needs one entry per tile (%d), got %d"
                             % (name, N_TILE, len(table)))
    adc_converter_mhz = 10 * ADC_SR / 1e6   # 2500, the rate before decimation
    fold_mhz = 2 * adc_converter_mhz        # a tone at 4905 comes back at 95
    for tile, (dac_nco, adc_nco) in enumerate(zip(DAC_NCO, ADC_NCO)):
        if abs(dac_nco + adc_nco - fold_mhz) > 1e-6:
            raise ValueError("tile %d: ADC_NCO should be %g - DAC_NCO"
                             % (tile, fold_mhz))

def validate_signal():
    """The CW tone has to stay inside the captured band and inside the DAC range."""
    if not 0 < CW_TONE_MHZ < ADC_SR / 2e6:
        raise ValueError("CW_TONE_MHZ must be between 0 and %g" % (ADC_SR / 2e6))
    if not 0 < CW_AMP <= 16383:
        raise ValueError("CW_AMP must be between 1 and 16383")
    if N_CAP <= 0 or N_CAP & (N_CAP - 1):
        raise ValueError("N_CAP must be a positive power of two, got %d" % N_CAP)
    if TRIG_HOLD_S <= 0:
        raise ValueError("TRIG_HOLD_S must be positive")

def validate_excitation():
    """Check the excitation method, and say what it is going to cost.

    A Hadamard matrix only exists for a power of two, so an array whose size
    is not one is padded up and pays the difference in extra bursts.
    """
    if not 90.0 <= INVERT_PHASE_DEG < 180.0:
        raise ValueError("INVERT_PHASE_DEG must be just under 180, got %g"
                         % INVERT_PHASE_DEG)
    n_elements = len(ALL_DACS)
    if not HADAMARD_EST:
        print("one-hot excitation: %d burst(s), one element live at a time"
              % n_elements)
        return
    order = 1
    while order < n_elements:
        order = order * 2
    print("hadamard excitation: %d burst(s) for %d element(s), +%.1f dB of SNR"
          % (order, n_elements, 10 * np.log10(order)))
    if order != n_elements:
        print("  %d element(s) is not a power of two, so %d burst(s) are padding"
              % (n_elements, order - n_elements))

validate_system_lists()
validate_channel_indices()
validate_converter_settings()
validate_signal()
CW_TONE_MHZ = snap_tone_to_fft_bin(CW_TONE_MHZ, ADC_SR, N_CAP)

ALL_DACS = DACS_0 + DACS_1   # the sweep order - a matrix column is a place in here
validate_excitation()

print("parameters ok - system 0: DAC %s ADC %s | system 1: DAC %s ADC %s"
      % (DACS_0, ADCS_0, DACS_1, ADCS_1))

## 3. Setup

Load the bitstream and tune every converter to the parameters above. Takes a few
seconds and only needs to happen once per kernel - and only one kernel at a time
may hold the overlay.

Everything that stays fixed for the whole sweep is set here, so the sweep itself
only ever writes the gain table. The DACs come up fully muted: nothing should
transmit until the sweep says so.

In [ ]:
import json
import os
import time

import numpy as np
import matplotlib.pyplot as plt

from lib.mts import doaMtsOverlay
from lib.common_functions import capture_aligned
from lib.common_functions import clear_dir
from lib.common_functions import convert_raw_to_iq
from lib.common_functions import create_tone_samples
from lib.common_functions import find_tone_bin
from lib.common_functions import save_json_params
from lib.common_functions import tune_adcs
from lib.common_functions import tune_dacs
from lib.common_functions import write_tone_to_players

def setup_overlay():
    """Load the overlay and tune it, so the sweep only has to change gains."""
    overlay = doaMtsOverlay("mts.bit")
    tune_dacs(overlay, DAC_NCO, DAC_ZONE, N_CH)
    tune_adcs(overlay, ADC_NCO, ADC_ZONE, N_CH, N_CAP)
    return overlay

overlay = setup_overlay()
print("overlay loaded, %d ADC channels open, all DACs muted" % N_CH)


## 4. Generate the CW tone

One harmonic at `CW_TONE_MHZ`, which section 2 already snapped onto an FFT bin.
The same waveform goes into all four player memories - only the gain decides
which DAC actually transmits, so the waveform is written once here and never
touched again during the sweep. That is what keeps the transmitted phase
identical from burst to burst.

In [ ]:
def generate_cw_tone(overlay):
    """Build the CW waveform and load it into all four DAC players."""
    n_samples = overlay.dac0_player.shape[0] // 2
    tone = create_tone_samples(n_samples, DAC_SR, CW_TONE_MHZ * 1e6, CW_AMP)
    write_tone_to_players(overlay, tone)
    return tone

tx_signal = generate_cw_tone(overlay)
print("CW tone loaded: %.6f MHz baseband, %d samples per player"
      % (CW_TONE_MHZ, len(tx_signal)))


## 5. Transmit one excitation pattern

A **pattern** is one row of the excitation table: one number per element of
`ALL_DACS`, saying how that element drives during this burst. Both methods are
just different tables, so the transmit code below never needs to know which one
is running.

- **one-hot** (`HADAMARD_EST = False`) - a single `1` and the rest `0`, so one
  element is live and its capture is directly its own column.
- **hadamard** (`HADAMARD_EST = True`) - every entry is `+1` or `-1`, so all the
  elements are live together and the capture is a signed sum of all of them.

The sign is the part the hardware cannot do directly: the gain register holds a
magnitude only. A `-1` is therefore sent as a phase flip, and because the
converter rejects exactly 180 degrees, `INVERT_PHASE_DEG` (179.99) is used
instead. The leftover hundredth of a degree is the same on every inverted
element, so it does not tilt one path against another.

Selection stays a gain-and-phase change, not a rewrite of the players, so the
transmitted waveform and its phase are the same in every burst.

One thing to watch with Hadamard: every element transmits at once, so the total
drive is `N` times the one-hot case. If anything in the chain compresses, the
capture stops being a linear sum and the un-mixing in section 8 will quietly
return wrong numbers - where one-hot would only have been noisier. If that is a
worry, scale the pattern rows down.

In [ ]:
def transmit_pattern(overlay, pattern):
    """Drive both systems with one excitation row, armed but idle."""
    overlay.d_gain = create_gain_table(pattern)
    overlay.d_phases = create_phase_table(pattern)
    overlay.configure_dacs()
    arm_dacs(overlay)

def create_gain_table(pattern):
    """Magnitude of each element's weight; every other DAC stays muted."""
    gains = [0.0] * N_CH
    for element, weight in enumerate(pattern):
        gains[ALL_DACS[element]] = abs(weight)
    return gains

def create_phase_table(pattern):
    """Send a negative weight as a phase flip - the gain has no sign.

    Exactly 180 degrees is rejected by the converter, so the flip is
    INVERT_PHASE_DEG instead.
    """
    phases = [0.0] * N_CH
    for element, weight in enumerate(pattern):
        if weight < 0:
            phases[ALL_DACS[element]] = INVERT_PHASE_DEG
    return phases

def arm_dacs(overlay):
    """Hand the player enable to trig_cap, so nothing plays until triggered."""
    overlay.dacs_off()
    overlay.da = 2


## 6. Receive

One aligned burst: TX and RX start on the same edge, so the phase of the capture
is a property of the channel and not of when the trigger happened to fire.

`capture_aligned()` is used instead of `get_custom_data_xm655()`, which would
fire its own trigger and undo the alignment.

The board hands back one flat int16 stream with all 32 lanes interleaved -
I and Q of ADC 0, then of ADC 1, and so on. `convert_raw_to_iq()` walks the
channels one at a time and pulls each pair out into a complex row.

Both live in `lib/common_functions.py`, so there is nothing to run in this section.

## 7. Vectorize with the FFT

A CW tone puts all of its energy in one bin, so a whole burst collapses to a
single complex number per ADC: the value of the FFT at the tone bin. That number
is the channel - its magnitude is the path gain, its angle is the path phase -
and it is far less noisy than any time domain estimate, because every bin that
is not the tone is dropped along with the noise it carried.

No window is applied. Section 2 put the tone exactly on a bin, so it does not
leak, and a window would only spread it into its neighbours.

The FFT is divided by the number of samples, so the result is an amplitude
instead of a number that grows with the capture length, and then by `CW_AMP`, so
each entry is what came out over what went in. That makes the matrices plain
gains: a magnitude of 0.1 means a path that loses 20 dB, whatever the DAC was
driven at.

In [ ]:
def extract_tone_vector(iq):
    """One complex channel coefficient per ADC, read at the tone bin.

    Divided by the transmitted amplitude, so an entry is what came out over
    what went in - a gain - and not a number that depends on how hard the
    DAC was driven.
    """
    n_samples = iq.shape[1]
    tone_bin = find_tone_bin(CW_TONE_MHZ, n_samples, ADC_SR)
    coefficients = np.zeros(N_CH, dtype=np.complex128)
    for channel in range(N_CH):
        spectrum = np.fft.fft(iq[channel])
        coefficients[channel] = spectrum[tone_bin] / n_samples / CW_AMP
    return coefficients


## 8. Sweep both systems and build the four matrices

All the DACs of a tile share one player memory, so they cannot be told apart by
what they send - only by *when* they send it, or by the signs they send it with.
That is the whole difference between the two methods, and both run as the same
loop: build the excitation table over `ALL_DACS`, fire one aligned burst per row,
then undo the mixing.

**One-hot.** The table is the identity: burst `k` has element `k` live and
everything else muted, so that capture already is column `k` and the un-mixing
step passes it straight through.

**Hadamard.** The table is a Hadamard matrix, so every burst has all the
elements live with `+1`/`-1` signs and every capture is a mixture. Because the
columns of a Hadamard matrix are orthogonal, combining the bursts back with the
same signs lines each element up with itself and cancels the others.
`unmix_measurements()` does exactly that: multiply by the pattern column, sum
over the bursts, divide by the column's energy. The result is the same channel,
measured with `N` times the energy, so **10·log₁₀(N) dB** better SNR at no extra
time - the burst count is the same.

A Hadamard matrix only exists for a power of two, so an array that is not one is
padded up to the next one; the extra columns are dropped and only cost bursts.

Every burst keeps all 16 coefficients, so `CHANNEL_LAYOUT` only picks rows and
columns out of the same measurements - the four matrices can never disagree
about what was going on.

| matrix | rows | columns |
|---|---|---|
| `H00` | `ADCS_0` | `DACS_0` |
| `H01` | `ADCS_0` | `DACS_1` |
| `H10` | `ADCS_1` | `DACS_0` |
| `H11` | `ADCS_1` | `DACS_1` |

In [ ]:
CHANNEL_LAYOUT = {
    "H00": (ADCS_0, DACS_0, "system 0 hearing itself - its self interference"),
    "H01": (ADCS_0, DACS_1, "system 1 reaching system 0"),
    "H10": (ADCS_1, DACS_0, "system 0 reaching system 1"),
    "H11": (ADCS_1, DACS_1, "system 1 hearing itself - its self interference")}

def estimate_channels(overlay):
    """Sweep every element of both systems and cut the four matrices out of it."""
    patterns = create_excitation_patterns()
    measurements = []
    for burst, pattern in enumerate(patterns):
        print("burst %d of %d: %s" % (burst + 1, len(patterns), describe(pattern)))
        measurements.append(measure_one_burst(overlay, pattern))
    overlay.dacs_off()
    coefficients = unmix_measurements(measurements, patterns)
    channels = {}
    for name, (adc_list, dac_list, description) in CHANNEL_LAYOUT.items():
        channels[name] = build_channel_matrix(coefficients, adc_list, dac_list)
    return channels

def create_excitation_patterns():
    """The excitation table: one row per burst, one column per element."""
    if HADAMARD_EST:
        return create_hadamard_patterns(len(ALL_DACS))
    return create_one_hot_patterns(len(ALL_DACS))

def create_one_hot_patterns(n_elements):
    """One element live per burst, the rest muted."""
    patterns = []
    for live in range(n_elements):
        row = [0.0] * n_elements
        row[live] = 1.0
        patterns.append(row)
    return patterns

def create_hadamard_patterns(n_elements):
    """Every element live in every burst, with Hadamard signs.

    The matrix is square and a power of two, so it is built at the next size
    up and the columns past the last element are simply not used.
    """
    order = calculate_hadamard_order(n_elements)
    matrix = create_hadamard_matrix(order)
    patterns = []
    for burst in range(order):
        row = []
        for element in range(n_elements):
            row.append(float(matrix[burst][element]))
        patterns.append(row)
    return patterns

def calculate_hadamard_order(n_elements):
    """Smallest power of two that holds every element."""
    order = 1
    while order < n_elements:
        order = order * 2
    return order

def create_hadamard_matrix(order):
    """Sylvester construction: keep doubling, negating the bottom right block."""
    matrix = np.ones((1, 1))
    while matrix.shape[0] < order:
        top = np.hstack([matrix, matrix])
        bottom = np.hstack([matrix, -matrix])
        matrix = np.vstack([top, bottom])
    return matrix

def measure_one_burst(overlay, pattern):
    """Transmit one pattern and return its 16 ADC coefficients."""
    transmit_pattern(overlay, pattern)
    raw = capture_aligned(overlay, TRIG_HOLD_S)
    iq = convert_raw_to_iq(raw, N_CH)
    return extract_tone_vector(iq)

def unmix_measurements(measurements, patterns):
    """Undo the excitation mixing: one column per element, per ADC.

    The columns of the excitation table are orthogonal, so weighting the
    bursts by one column and adding them up keeps that element and cancels
    every other. Dividing by the column's energy takes the scale back out.
    With a one-hot table this is a no-op - burst k already was element k.
    """
    n_elements = len(patterns[0])
    coefficients = np.zeros((N_CH, n_elements), dtype=np.complex128)
    for element in range(n_elements):
        energy = 0.0
        for burst, pattern in enumerate(patterns):
            coefficients[:, element] += pattern[element] * measurements[burst]
            energy += pattern[element] ** 2
        coefficients[:, element] /= energy
    return coefficients

def build_channel_matrix(coefficients, adc_list, dac_list):
    """Take the rows of one receive set and the columns of one transmit set.

    A column of coefficients is a place in ALL_DACS, so a DAC index has to be
    looked up and cannot be used directly.
    """
    matrix = np.zeros((len(adc_list), len(dac_list)), dtype=np.complex128)
    for row, adc_index in enumerate(adc_list):
        for column, dac_index in enumerate(dac_list):
            matrix[row, column] = coefficients[adc_index, ALL_DACS.index(dac_index)]
    return matrix

def describe(pattern):
    """The live elements of the pattern as a short string, for the progress line."""
    parts = []
    for element, weight in enumerate(pattern):
        if weight:
            parts.append("DAC %d %+.0f" % (ALL_DACS[element], weight))
    return ", ".join(parts)

CHANNELS = estimate_channels(overlay)
for name, matrix in CHANNELS.items():
    print("%s %s" % (name, matrix.shape))

## 9. Save

The four matrices go to `output/channels/` as `h00.npy`, `h01.npy`, `h10.npy` and
`h11.npy`, plus a `params.json` saying which DAC each column is and which ADC each
row is - without it the matrices are anonymous.

The folder is emptied first. A leftover file from an earlier run carries the
same name but a different shape and a different meaning, and nothing downstream
can tell the two apart, so a half-overwritten `output/channels/` is worse than no
`output/channels/` at all. Only the files sitting directly in the folder are removed -
a subfolder is left alone, because the thing to avoid is a mixed old-and-new
result, not somebody else's data.

`complex64` is enough: the samples are 14 bit integers, so nothing is lost and
the files are half the size.

In [ ]:
def save_channels(channels, folder="output/channels"):
    """Replace the channels folder with the matrices from this run."""
    clear_dir(folder)
    for name, matrix in channels.items():
        path = os.path.join(folder, "%s.npy" % name.lower())
        np.save(path, matrix.astype(np.complex64))
    save_channel_params(folder)
    return folder

def save_channel_params(folder):
    """Record what the rows and the columns of the matrices stand for."""
    params = {"dacs_0": DACS_0, "adcs_0": ADCS_0,
              "dacs_1": DACS_1, "adcs_1": ADCS_1,
              "dac_nco": DAC_NCO, "dac_zone": DAC_ZONE,
              "adc_nco": ADC_NCO, "adc_zone": ADC_ZONE,
              "cw_tone_mhz": CW_TONE_MHZ, "cw_amp": CW_AMP,
              "hadamard_est": HADAMARD_EST,
              "invert_phase_deg": INVERT_PHASE_DEG,
              "n_cap": N_CAP, "dac_sr": DAC_SR, "adc_sr": ADC_SR}
    save_json_params(folder, params)

print("saved ->", save_channels(CHANNELS))

## 10. Look at the result

A sanity check before the matrices get used anywhere. Each entry is printed as a
magnitude in dB and a phase in degrees, one line per path, and then drawn as two
small images.

What to expect: the self interference matrices `H00` and `H11` should sit well
above the cross ones. If they do not, the tone went out of a DAC that is not
connected to the antennas - check `DACS_0` and `DACS_1`, since a path 30 dB down
is crosstalk and its phase is noise.

In [ ]:
def print_channel_matrix(matrix, adc_list, dac_list, title):
    """One line per path, so the numbers can be read without squinting."""
    print(title)
    for row, adc_index in enumerate(adc_list):
        for column, dac_index in enumerate(dac_list):
            value = matrix[row, column]
            print("  DAC %2d -> ADC %2d : %7.2f dB  %+8.2f deg"
                  % (dac_index, adc_index, 20 * np.log10(np.abs(value) + 1e-12),
                     np.rad2deg(np.angle(value))))

def plot_channel_matrix(matrix, adc_list, dac_list, title):
    """Magnitude and phase of one channel matrix, side by side."""
    magnitude_db = 20 * np.log10(np.abs(matrix) + 1e-12)
    phase_deg = np.rad2deg(np.angle(matrix))
    figure, axes = plt.subplots(1, 2, figsize=(9, 3.2))
    panels = [(magnitude_db, "magnitude (dB)"), (phase_deg, "phase (deg)")]
    for axis, (data, label) in zip(axes, panels):
        image = axis.imshow(data, aspect="auto")
        axis.set_title(label, fontsize=9)
        axis.set_xticks(range(len(dac_list)), ["DAC %d" % d for d in dac_list])
        axis.set_yticks(range(len(adc_list)), ["ADC %d" % a for a in adc_list])
        figure.colorbar(image, ax=axis)
    figure.suptitle(title)
    plt.tight_layout()
    plt.show()

for name, (adc_list, dac_list, description) in CHANNEL_LAYOUT.items():
    print_channel_matrix(CHANNELS[name], adc_list, dac_list,
                         "%s - %s" % (name, description))

for name, (adc_list, dac_list, description) in CHANNEL_LAYOUT.items():
    plot_channel_matrix(CHANNELS[name], adc_list, dac_list,
                        "%s - %s" % (name, description))


## 11. Stop

Switch the transmitter off when you are done.

In [ ]:
overlay.dacs_off()
